# ColliderFM Dataset And Dataloader Walkthrough

If you just joined the project, start here.

This notebook stays on the raw-data side of the pipeline and answers the practical questions that usually come up first:

- what exactly comes back from ColliderML on Hugging Face?
- what does one event look like before we build any point views?
- what schema cleanup does `ColliderMLDataset` apply before view building?
- what does the dataloader hand to the training code?

The idea is to make the data contract obvious before we start talking about augmentations, backbones, or SSL losses.


## Where This Fits

Right now the runtime path is intentionally simple and calo-only.

That means:

- we load `calo_hits`
- we keep the raw calorimeter `total_energy` field and defer feature transforms to `views.py`
- we keep raw events as Python dictionaries for as long as possible
- we only build model-facing point views after batching

That separation makes it easier to debug data issues without mixing them up with model issues.


In [ ]:
import inspect
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import torch

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_ROOT = PROJECT_ROOT / 'src'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from collider_fm.project_config import load_project_config
from collider_fm.data import CALO_DATASET_ENERGY_KEY, ColliderMLDataset, collate_fn
from torch.utils.data import DataLoader

plt.style.use('default')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = False


## Configuration

The defaults keep the notebook lightweight, but still realistic enough to show the shapes and ranges we care about.

One detail worth noticing if you're new to Hugging Face datasets: split strings like `train[:8]` are handled directly by the dataset loader, so they are an easy way to keep notebook exploration fast.


In [ ]:
SEED = 7
PROJECT_CONFIG = load_project_config()
DATA_CONFIG = PROJECT_CONFIG.data

DATASET_NAME = DATA_CONFIG.dataset_name
DATASET_TYPE = DATA_CONFIG.dataset_type
PU_CONFIG = DATA_CONFIG.pu_config
OBJECT_TYPES = ["calo_hits", "particles"]  # list(DATA_CONFIG.object_types)
TRAIN_SPLIT = 'train[:8]'
DETAIL_INDEX = 0
BATCH_SIZE = 2
CACHE_DIR = DATA_CONFIG.cache_dir
DATASET_REVISION = DATA_CONFIG.dataset_revision
LOCAL_FILES_ONLY = DATA_CONFIG.local_files_only

np.random.seed(SEED)
torch.manual_seed(SEED)

print('Project root:', PROJECT_ROOT)
print('Dataset name:', DATASET_NAME)
print('Dataset config:', f'{DATASET_TYPE}_{PU_CONFIG}_calo_hits')
print('Split:', TRAIN_SPLIT)
print('Batch size:', BATCH_SIZE)
print('Dataset revision:', DATASET_REVISION)
print('Local files only:', LOCAL_FILES_ONLY)
print('Energy key:', CALO_DATASET_ENERGY_KEY)


## Load The Dataset

`ColliderMLDataset` is intentionally small. It does three main things for us:

- loads the requested ColliderML configuration
- keeps variable-length events intact
- keeps the raw calorimeter energy field as `total_energy`


## Read The Dataset Source

This is worth reading directly because most of the project-specific split logic lives there.


In [ ]:
dataset = ColliderMLDataset(
    dataset_name=DATASET_NAME,
    dataset_type=DATASET_TYPE,
    pu_config=PU_CONFIG,
    object_types=OBJECT_TYPES,
    split=TRAIN_SPLIT,
    cache_dir=CACHE_DIR,
    dataset_revision=DATASET_REVISION,
    local_files_only=LOCAL_FILES_ONLY,
)

print('Dataset length:', len(dataset))
print('Dataset object types:', dataset.object_types)


In [ ]:
print(inspect.getsource(ColliderMLDataset))


## Inspect One Raw Event

At this point nothing has been turned into a point cloud yet. We are still looking at the raw nested event dictionary.


In [ ]:
event = dataset[DETAIL_INDEX]
calo_hits = event['calo_hits']

print('Top-level keys:', list(event.keys()))
print('Calo fields:', sorted(calo_hits.keys()))
print('Number of calorimeter hits:', len(calo_hits['x']))
print('First five energies:', calo_hits['total_energy'][:5])


In [ ]:
event = dataset[DETAIL_INDEX]
particles = event['particles']

print('Top-level keys:', list(event.keys()))
print('Particle fields:', sorted(particles.keys()))
print('Number of particles:', len(particles['px']))
print('First five energies:', particles['energy'][:5])
print("unique PDG IDs:", np.unique(particles['pdg_id']))

In [ ]:
def tensor_preview(value: Any, rows: int = 5) -> Any:
    if isinstance(value, torch.Tensor):
        return value[:rows].cpu().tolist()
    return value

raw_event_summary = {
    'field_shapes': {
        key: list(value.shape) if isinstance(value, torch.Tensor) else type(value).__name__
        for key, value in calo_hits.items()
    },
    'energy_key_check': CALO_DATASET_ENERGY_KEY in calo_hits,
    'preview': {
        key: tensor_preview(value)
        for key, value in calo_hits.items()
        if key in {'x', 'y', 'z', 'total_energy'}
    },
}
raw_event_summary


## Plot One Raw Event

This is the quickest way to get a feel for the data before we start sampling hits or building views.


In [ ]:
coord = torch.stack([calo_hits['z'], calo_hits['x'], calo_hits['y']], dim=1).cpu().numpy()
energy = calo_hits['total_energy'].cpu().numpy()
positive_energy = energy[energy > 0]

if positive_energy.size > 0:
    vmin = float(positive_energy.min())
    vmax = float(np.quantile(positive_energy, 0.99))
    vmax = max(vmax, vmin * 1.01)
    marker_size = np.clip(np.log10(energy / vmin + 1.0) * 10.0, 4.0, 60.0)
    norm = LogNorm(vmin=vmin, vmax=vmax)
else:
    marker_size = np.full_like(energy, 4.0)
    norm = None

fig = plt.figure(figsize=(12, 5))
ax0 = fig.add_subplot(1, 2, 1, projection='3d')
scatter = ax0.scatter(coord[:, 0], coord[:, 1], coord[:, 2], c=energy, s=marker_size, cmap='inferno', alpha=0.6, norm=norm)
fig.colorbar(scatter, ax=ax0, shrink=0.7, pad=0.1, label='energy')
ax0.set_title('Raw calorimeter geometry')
ax0.set_xlabel('z')
ax0.set_ylabel('x')
ax0.set_zlabel('y')

ax1 = fig.add_subplot(1, 2, 2)
ax1.hist(energy, bins=50, color='tab:red', alpha=0.85)
ax1.set_title('Energy histogram')
ax1.set_xlabel('energy')
ax1.set_ylabel('count')
plt.tight_layout()
plt.show()


## Small-Sample Dataset Stats

Looking at a handful of events is often enough to spot obvious issues with hit counts, scales, or weird tails.


In [ ]:
hit_counts = []
energy_means = []
energy_maxes = []
radius_means = []

for sample in dataset:
    hits = sample['calo_hits']
    coord_xyz = torch.stack([hits['x'], hits['y'], hits['z']], dim=1)
    radius = torch.linalg.norm(coord_xyz, dim=1)
    hit_counts.append(int(len(hits['x'])))
    energy_means.append(float(hits['total_energy'].mean().item()))
    energy_maxes.append(float(hits['total_energy'].max().item()))
    radius_means.append(float(radius.mean().item()))

sample_stats = {
    'num_events': len(hit_counts),
    'hit_count_min': int(np.min(hit_counts)),
    'hit_count_max': int(np.max(hit_counts)),
    'hit_count_mean': float(np.mean(hit_counts)),
    'mean_energy_mean': float(np.mean(energy_means)),
    'max_energy_mean': float(np.mean(energy_maxes)),
    'radius_mean_mean': float(np.mean(radius_means)),
}
sample_stats


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(hit_counts, bins=min(len(hit_counts), 8), color='tab:blue', alpha=0.85)
axes[0].set_title('Hits per event')
axes[1].hist(energy_means, bins=min(len(energy_means), 8), color='tab:orange', alpha=0.85)
axes[1].set_title('Mean event energy')
axes[2].hist(radius_means, bins=min(len(radius_means), 8), color='tab:green', alpha=0.85)
axes[2].set_title('Mean hit radius')
plt.tight_layout()
plt.show()


## Inspect The Dataloader

The dataloader is intentionally boring here, and that is a good thing.

Because ColliderML events have different numbers of hits, the custom `collate_fn` returns a list of event dictionaries instead of padding or packing them into a rectangular tensor too early.


In [ ]:
print(inspect.getsource(collate_fn))


In [ ]:
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
batch = next(iter(dataloader))

print('Batch type:', type(batch).__name__)
print('Number of events in batch:', len(batch))
print('Event 0 top-level keys:', list(batch[0].keys()))
print('Event 0 calo hit count:', len(batch[0]['calo_hits']['x']))
print('Event 1 calo hit count:', len(batch[1]['calo_hits']['x']))


In [ ]:
dataloader_summary = {
    'batch_container_type': type(batch).__name__,
    'event_hit_counts': [int(len(item['calo_hits']['x'])) for item in batch],
    'energy_alias_checks': [
        CALO_DATASET_ENERGY_KEY in item['calo_hits']
        for item in batch
    ],
}
dataloader_summary


## Why We Keep The Dataloader Simple

A quick mental model for the division of labor in the repo:

- `ColliderMLDataset` loads events and fixes small schema differences
- `collate_fn` preserves variable-length events
- `src/collider_fm/views.py` turns raw events into model-ready point views
- `scripts/train.py` builds teacher and student views after loading a batch

That separation is one of the reasons the pipeline is still pretty easy to reason about.


## Next Stop

From here, the next useful notebook is `notebooks/model_walkthrough.ipynb`.

That one picks up where this one stops and walks through:

- raw event -> point view
- point view -> augmentations
- augmentations -> student/teacher batch
- model outputs -> SSL loss and validation plots
